In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cross_decomposition import PLSRegression

In [2]:
z_score_df = pd.read_pickle('matrix_Lisbon_threshold.pkl')

In [3]:
import pickle

with open('list_groups.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT']


In [4]:
list_groups = pd.Series(list_groups)

In [5]:
def calculate_vip(pls, X):
    
    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_
    
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    
    Wnorm = (w ** 2) / np.sum(w ** 2, axis=0)
    
    vip = np.sqrt(p * (Wnorm @ s) / np.sum(s))
    
    return vip.flatten()

In [6]:
def stratified_bootstrap(X, y):
    """
    Creates a bootstrap sample preserving the original class proportions.
    """
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y.iloc[indices]


# Settings
n_iterations = 100
feature_names = z_score_df.columns

importance_matrix = np.zeros((n_iterations, len(feature_names)))
ranking_matrix = np.zeros((n_iterations, len(feature_names)))

vip_matrix = np.zeros((n_iterations, len(feature_names)))
vip_ranking_matrix = np.zeros((n_iterations, len(feature_names)))
vip_selection_matrix = np.zeros((n_iterations, len(feature_names)))

le = LabelEncoder()
le.fit(list_groups)


# Main loop
for i in range(n_iterations):

    X_boot, y_boot = stratified_bootstrap(z_score_df, list_groups)

    # =====================
    # RANDOM FOREST 
    # =====================

    rf = RandomForestClassifier(
        n_estimators=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=i
    )

    rf.fit(X_boot, y_boot)

    # importances
    importances = rf.feature_importances_
    importance_matrix[i] = importances

    # ranking (1 = migliore)
    ranks = (-importances).argsort().argsort() + 1
    ranking_matrix[i] = ranks

    # =====================
    # PLS-DA 
    # =====================

    # encode y
    y_encoded = le.transform(y_boot)

    pls = PLSRegression(n_components=2)
    pls.fit(X_boot, y_encoded)

    # VIP
    vip_scores = calculate_vip(pls, X_boot)
    vip_matrix[i] = vip_scores

    # ranking VIP (1 = migliore)
    vip_ranks = (-vip_scores).argsort().argsort() + 1
    vip_ranking_matrix[i] = vip_ranks
    vip_selection_matrix[i] = (vip_scores > 1).astype(int)


# Aggregazione risultati
mean_importance = importance_matrix.mean(axis=0)
mean_rank = ranking_matrix.mean(axis=0)
std_rank = ranking_matrix.std(axis=0)

mean_vip = vip_matrix.mean(axis=0)
mean_vip_rank = vip_ranking_matrix.mean(axis=0)
std_vip_rank = vip_ranking_matrix.std(axis=0)
vip_freq = vip_selection_matrix.mean(axis=0)


# DataFrame finale
importance_df = pd.DataFrame({
    "protein": feature_names,
    "importance": mean_importance,
    "mean_rank": mean_rank,
    "rank_std": std_rank
}).sort_values(by="importance", ascending=False)

vip_df = pd.DataFrame({
    "protein": feature_names,
    "VIP": mean_vip,
    "mean_rank_vip": mean_vip_rank,
    "std_rank_vip": std_vip_rank,
    "vip_freq": vip_freq
}).sort_values(by="VIP", ascending=False)

In [7]:
vip_df

,protein,VIP,mean_rank_vip,std_rank_vip,vip_freq
133,P14618,2.650248,1.00,0.000000,1.00
164,P40925,2.175581,3.81,2.741149,1.00
238,Q9NQ79,2.002914,7.20,5.194228,1.00
191,Q13449,1.965987,8.14,5.943097,1.00
194,Q14118,1.916366,10.29,8.163694,1.00
...,...,...,...,...,...
149,P23471,0.390829,217.77,41.750414,0.01
221,Q8TEU8,0.383045,220.53,35.283553,0.00
7,O14773,0.377914,220.74,41.879976,0.01
226,Q92876,0.376865,218.98,42.378528,0.01


In [8]:
importance_df

,protein,importance,mean_rank,rank_std
133,P14618,0.066461,1.33,0.693614
191,Q13449,0.041265,4.12,2.695478
164,P40925,0.036177,6.07,4.652429
238,Q9NQ79,0.034542,7.19,8.331500
194,Q14118,0.026408,11.53,12.648680
...,...,...,...,...
55,P02679,0.000682,203.13,46.343210
43,P01857,0.000663,206.48,41.583766
84,P05090,0.000652,207.71,41.753873
29,P00751,0.000647,208.06,44.523661


In [9]:
final_df = importance_df.merge(vip_df, on="protein")
final_df["rank_diff"] = abs(final_df["mean_rank"] - final_df["mean_rank_vip"])
final_df["mean_rank_combined"] = (
    final_df["mean_rank"] + final_df["mean_rank_vip"]
) / 2
final_df["std_rf"] = final_df["rank_std"]
final_df["std_pls"] = final_df["std_rank_vip"]
final_df = final_df[[
    "protein",
    "mean_rank_combined",
    "std_rf",
    "std_pls",
    "vip_freq",
    "rank_diff"
]]
final_df = final_df.sort_values(by="mean_rank_combined")

In [10]:
final_df.head(20)

,protein,mean_rank_combined,std_rf,std_pls,vip_freq,rank_diff
0,P14618,1.165,0.693614,0.000000,1.00,0.33
2,P40925,4.940,4.652429,2.741149,1.00,2.26
1,Q13449,6.130,2.695478,5.943097,1.00,4.02
3,Q9NQ79,7.195,8.331500,5.194228,1.00,0.01
4,Q14118,10.910,12.648680,8.163694,1.00,1.24
6,P07195,12.110,10.653352,8.902915,1.00,1.40
5,P02747,12.270,9.818651,11.791569,1.00,1.12
10,P25311,16.385,16.974484,9.881801,1.00,8.97
12,P07333,16.405,16.938465,14.837422,0.99,6.75
8,P02766,17.425,8.878153,14.954367,1.00,9.41
